# 01 — Data collection
Three raw sources → combined feature frame for modeling:
1. Yahoo Finance daily OHLCV (price base, ~53 EU mid-caps, 2005–today)
2. FRED macro series (rates, VIX, USD/EUR)
3. Per-ticker news headlines via Google News RSS (alt-data event source)

Finally we run the feature builder which writes `data/processed/df_combined.parquet` for notebooks 02–04.

In [1]:
# Kernel bootstrap — run first. Finds project root whether Jupyter was launched
# from notebooks/ or from the repo root; installs any missing packages.
import sys, os
def _find_root():
    """Walk up from cwd looking for the folder that contains src/ + requirements.txt."""
    for base in [os.getcwd(), os.path.abspath(".."), os.path.abspath(".")]:
        for cand in [base, os.path.dirname(base)]:
            if os.path.isfile(os.path.join(cand, "requirements.txt")) and os.path.isdir(os.path.join(cand, "src")):
                return cand
    return os.path.abspath("..")
_project_root = _find_root()
if _project_root not in sys.path: sys.path.insert(0, _project_root)
os.chdir(_project_root)
from src.bootstrap import ensure
ensure({'pandas_datareader':'pandas-datareader','ta':'ta','yaml':'pyyaml',
        'yfinance':'yfinance','xgboost':'xgboost','optuna':'optuna',
        'dotenv':'python-dotenv'})
print("bootstrap OK. project_root =", _project_root)


bootstrap OK. project_root = /home/yazadanparast/Downloads/eu-midcap-echo


In [ ]:
%load_ext autoreload
%autoreload 2
# 1) Yahoo Finance prices (first run: 196 tickers × 20 years; cached after that)
from src.data.download_yf import download as dl_yf
prices = dl_yf()
prices.head()

12:59:48 [INFO] eu-midcap-echo:   yf download 1COV.DE from 2005-01-01
12:59:49 [INFO] eu-midcap-echo:   yf download ABB.ST from 2005-01-01
12:59:50 [INFO] eu-midcap-echo:   yf download ABBN.SW from 2005-01-01
12:59:52 [INFO] eu-midcap-echo:   yf download ABI.BR from 2005-01-01
12:59:53 [INFO] eu-midcap-echo:   yf download ACA.PA from 2005-01-01
12:59:54 [INFO] eu-midcap-echo:   yf download ACS.MC from 2005-01-01
12:59:55 [INFO] eu-midcap-echo:   yf download AD.AS from 2005-01-01
12:59:57 [INFO] eu-midcap-echo:   yf download ADS.DE from 2005-01-01
12:59:58 [INFO] eu-midcap-echo:   yf download ADYEN.AS from 2005-01-01
12:59:59 [INFO] eu-midcap-echo:   yf download AFX.DE from 2005-01-01
13:00:00 [INFO] eu-midcap-echo:   yf download AI.PA from 2005-01-01
13:00:01 [INFO] eu-midcap-echo:   yf download AIR.DE from 2005-01-01
13:00:02 [INFO] eu-midcap-echo:   yf download AIR.PA from 2005-01-01
13:00:04 [INFO] eu-midcap-echo:   yf download AIXA.DE from 2005-01-01
13:00:05 [INFO] eu-midcap-echo:

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Ticker
0,2026-10-19,59.099998,59.700001,59.099998,59.700001,59.700001,4048,0.0,0.0,1COV.DE
1,2026-10-20,60.299999,60.299999,60.000000,60.099998,60.099998,2739,0.0,0.0,1COV.DE
2,2026-10-21,60.200001,60.900002,60.200001,60.900002,60.900002,2102,0.0,0.0,1COV.DE
3,2026-10-22,60.200001,61.200001,60.099998,60.299999,60.299999,8438,0.0,0.0,1COV.DE
4,2026-10-23,60.299999,60.500000,60.099998,60.200001,60.200001,2508,0.0,0.0,1COV.DE


In [3]:
%load_ext autoreload
%autoreload 2
# 2) FRED macro series 
from src.data.download_fred import download as dl_fred
macro = dl_fred()
macro.tail()

13:03:48 [INFO] eu-midcap-echo: fred download 7 series


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


13:03:55 [INFO] eu-midcap-echo: macro: 5,783 rows


,Date,us_10y,us_5y,us_1y,fed_funds,ted_spread,vix,usd_eur
5778,2026-11-12,5.10,4.95,4.65,NaN,NaN,18.70,1.1056
5779,2026-11-13,5.15,4.98,4.65,NaN,NaN,19.30,1.1034
5780,2026-11-16,5.12,4.93,4.62,NaN,NaN,22.91,NaN
5781,2026-11-17,5.10,4.90,4.60,NaN,NaN,25.46,NaN
5782,2026-11-18,5.07,4.89,4.62,NaN,NaN,24.56,NaN


In [4]:
%load_ext autoreload
%autoreload 2
# 3) Per-ticker news headlines via Google News RSS (parquet-cached)
from src.data.download_dgap import download as dl_news
news = dl_news()
news.head(10)

13:03:55 [INFO] eu-midcap-echo:   news 1COV.DE (1/196)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


13:03:56 [INFO] eu-midcap-echo:   news ABB.ST (2/196)
13:03:57 [INFO] eu-midcap-echo:   news ABBN.SW (3/196)
13:03:59 [INFO] eu-midcap-echo:   news ABI.BR (4/196)
13:04:00 [INFO] eu-midcap-echo:   news ACA.PA (5/196)
13:04:01 [INFO] eu-midcap-echo:   news ACS.MC (6/196)
13:04:03 [INFO] eu-midcap-echo:   news AD.AS (7/196)
13:04:04 [INFO] eu-midcap-echo:   news ADS.DE (8/196)
13:04:06 [INFO] eu-midcap-echo:   news ADYEN.AS (9/196)
13:04:07 [INFO] eu-midcap-echo:   news AFX.DE (10/196)
13:04:09 [INFO] eu-midcap-echo:   news AI.PA (11/196)
13:04:10 [INFO] eu-midcap-echo:   news AIR.DE (12/196)
13:04:12 [INFO] eu-midcap-echo:   news AIR.PA (13/196)
13:04:13 [INFO] eu-midcap-echo:   news AIXA.DE (14/196)
13:04:15 [INFO] eu-midcap-echo:   news AKZA.AS (15/196)
13:04:16 [INFO] eu-midcap-echo:   news ALV.DE (16/196)
13:04:17 [INFO] eu-midcap-echo:   news AMS.MC (17/196)
13:04:19 [INFO] eu-midcap-echo:   news ANDR.VI (18/196)
13:04:20 [INFO] eu-midcap-echo:   news ASM.AS (19/196)
13:04:21 [INFO

,Date,Ticker,title,body,url
0,2026-11-08,1COV.DE,Covestro (DB:1COV) Stock May Look Undervalued ...,,https://news.google.com/rss/articles/CBMilgFBV...
1,2026-06-17,1COV.DE,Covestro Number of Employees 2026 | Employee C...,,https://news.google.com/rss/articles/CBMiZ0FVX...
2,2026-09-28,1COV.DE,BASF's bid for Evonik rejected: €10.3 billion ...,,https://news.google.com/rss/articles/CBMidkFVX...
3,2026-11-10,1COV.DE,Covestro stock slips 0.58% below Tuesday's Xet...,,https://news.google.com/rss/articles/CBMitwFBV...
4,2026-10-20,1COV.DE,Covestro consensus target stands at EUR 62.00:...,,https://news.google.com/rss/articles/CBMi5AFBV...
5,2026-10-20,ABB.ST,AI worries weigh on ABB shares despite record ...,,https://news.google.com/rss/articles/CBMitgFBV...
6,2026-01-29,ABB.ST,ABB Earnings: Strong Finish to 2025 Supports P...,,https://news.google.com/rss/articles/CBMipAFBV...
7,2026-07-22,ABB.ST,ABB (SWX:ABBN) Stock Looks Above Fair Value Af...,,https://news.google.com/rss/articles/CBMikwFBV...
8,2026-10-07,ABB.ST,"ABB Stock: Strong Execution, Expensive Expecta...",,https://news.google.com/rss/articles/CBMijAFBV...
9,2026-04-22,ABB.ST,ABB shares surge as data centre business boost...,,https://news.google.com/rss/articles/CBMiqgFBV...


## Build the combined feature frame
Joins prices + macro + news → 150+ features (TA, momentum, volatility, sentiment, macro, categorical dummies). Writes `data/processed/df_combined.parquet`.

In [5]:
%load_ext autoreload
%autoreload 2
from src.features.build_features import build_dataset
df, dummies = build_dataset()
print('df_combined shape:', df.shape)
print('non-null target:', df.is_positive_growth_20d_future.notna().sum())
df.head(3)

13:08:19 [INFO] eu-midcap-echo: loading raw yf_all
13:08:19 [INFO] eu-midcap-echo:   prices: 994,718 rows


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


13:08:20 [INFO] eu-midcap-echo: applying TA features
13:08:36 [INFO] eu-midcap-echo:   after TA: (994718, 104)
13:10:38 [INFO] eu-midcap-echo:   after macro: (994718, 116)
13:10:40 [INFO] eu-midcap-echo:   after sentiment: (994718, 128)
13:10:56 [INFO] eu-midcap-echo: saved df_combined.parquet  shape=(994718, 178)


df_combined shape: (994718, 178)
non-null target: 994718


,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Ticker,...,sector_retail,sector_semis,sector_shipping,sector_steel,sector_tech,sector_telecom,sector_utilities,disclosure_polarity_neg,disclosure_polarity_none,disclosure_polarity_pos
0,2026-10-19,59.099998,59.700001,59.099998,59.700001,59.700001,4048,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
1,2026-10-20,60.299999,60.299999,60.000000,60.099998,60.099998,2739,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
2,2026-10-21,60.200001,60.900002,60.200001,60.900002,60.900002,2102,0.0,0.0,1COV.DE,...,0,0,0,0,0,0,0,0,1,0
